# PredictShield AI - Fusion Layer (5-Model)

Trains the meta-learner MLP that fuses ALL 5 models into one verdict.

**Fusion input (47 dims):**
- xgb_proba (15)
- lstm_proba (15)
- transformer_proba (15)
- if_anomaly_score (1)
- gnn_source_ip_risk (1)  <- "is the SOURCE device compromised?" (GNN node risk)

**Data alignment:** `training_ready.csv` (77 features, no IPs) and `generated_combined.csv` (has Source IP) have the SAME row order (2.8M rows). So flow i in training_ready = flow i in generated. We track each sequence's last-flow index, then look up its Source IP in generated, then map to GNN node risk.

**All models + datasets loaded from Kaggle INPUT** (not /kaggle/working).

In [1]:
import sys, subprocess
# Install missing deps (torch-geometric + onnxruntime)
try:
    import torch, torch_geometric
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch-geometric", "--no-deps"])
    import torch, torch_geometric
try:
    import onnxruntime
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "onnxruntime", "onnx"])
    import onnxruntime
# --- all imports (moved here) ---
import numpy as np
import pandas as pd
import joblib
import onnxruntime as ort
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch_geometric.nn import GATConv
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report

print(f'PyTorch {torch.__version__}  PyG {torch_geometric.__version__}  onnxruntime {onnxruntime.__version__}')


PyTorch 2.11.0+cu130  PyG 2.8.1  onnxruntime 1.31.1


In [2]:
WINDOW = 50
STRIDE = 10
N_FEATURES = 77
N_CLASSES = 15
FUSION_DIM = 47  # 15+15+15+1+1
NODE_FEAT_DIM = 16
GNN_HIDDEN = 32
EPOCHS = 50
LR = 0.001
BATCH_SIZE = 256
RANDOM_STATE = 42

torch.manual_seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


Device: cuda


In [3]:
# ALL models + datasets from Kaggle INPUT (exact paths)
XGB_ONNX = '/kaggle/input/models/pritammandal143/xgboost-cicids2017-v2/onnx/default/1/xgboost_cicids2017_v2.onnx'
LSTM_ONNX = '/kaggle/input/models/pritammandal143/lstm-cicids2017/onnx/default/1/lstm_cicids2017.onnx'
TRANS_ONNX = '/kaggle/input/models/pritammandal143/transformer-cicids2017/onnx/default/1/transformer_cicids2017.onnx'
IF_PKL = '/kaggle/input/models/pritammandal143/isolation-forest-cicids2017-v2/scikitlearn/default/1/isolation_forest_cicids2017_v2.pkl'
GNN_PT = '/kaggle/input/models/pritammandal143/gat-attack-path-state/pytorch/default/1/gat_attack_path_state.pt'

TR_CSV = '/kaggle/input/datasets/pritammandal143/training/cicids2017_training_ready.csv'
GEN_CSV = '/kaggle/input/datasets/pritammandal143/cicids2017-generated/cicids2017_generated_combined.csv'

import os
for f in [XGB_ONNX, LSTM_ONNX, TRANS_ONNX, IF_PKL, GNN_PT, TR_CSV, GEN_CSV]:
    print(f'{"OK" if os.path.exists(f) else "MISSING"}  {f}')


OK  /kaggle/input/models/pritammandal143/xgboost-cicids2017-v2/onnx/default/1/xgboost_cicids2017_v2.onnx
OK  /kaggle/input/models/pritammandal143/lstm-cicids2017/onnx/default/1/lstm_cicids2017.onnx
OK  /kaggle/input/models/pritammandal143/transformer-cicids2017/onnx/default/1/transformer_cicids2017.onnx
OK  /kaggle/input/models/pritammandal143/isolation-forest-cicids2017-v2/scikitlearn/default/1/isolation_forest_cicids2017_v2.pkl
OK  /kaggle/input/models/pritammandal143/gat-attack-path-state/pytorch/default/1/gat_attack_path_state.pt
OK  /kaggle/input/datasets/pritammandal143/training/cicids2017_training_ready.csv
OK  /kaggle/input/datasets/pritammandal143/cicids2017-generated/cicids2017_generated_combined.csv


In [4]:
# Load training_ready (scaled 77 features) + generated (Source IP)
tr = pd.read_csv(TR_CSV, dtype=np.float32)
tr['label'] = tr['label'].astype(np.int32)
feature_cols = [c for c in tr.columns if c != 'label']

# Downsample BENIGN, KEEP original indices for IP lookup later
BENIGN_TARGET = 200_000
y_all = tr['label'].values
benign_idx = np.where(y_all == 0)[0]
attack_idx = np.where(y_all != 0)[0]
np.random.seed(RANDOM_STATE)
benign_sampled = np.random.choice(benign_idx, size=BENIGN_TARGET, replace=False)
keep_idx = np.sort(np.concatenate([benign_sampled, attack_idx]))

X = tr[feature_cols].values[keep_idx].astype(np.float32)
y = y_all[keep_idx].astype(np.int64)
print(f'Downsampled: {len(X):,} rows, original indices kept')
del tr


Downsampled: 756,556 rows, original indices kept


In [5]:
# Build sequences, tracking last-flow ORIGINAL index (keep_idx mapping)
def build_sequences(X, y, orig_idx, window=WINDOW, stride=STRIDE):
    seqs, labels, last_idx = [], [], []
    for start in range(0, len(X) - window + 1, stride):
        seqs.append(X[start:start + window])
        labels.append(y[start + window - 1])
        last_idx.append(orig_idx[start + window - 1])  # ORIGINAL row index
    return np.stack(seqs), np.array(labels), np.array(last_idx)

X_seq, y_seq, last_idx = build_sequences(X, y, keep_idx)
print(f'Sequences: {X_seq.shape}')

# Drop classes with < 5 sequences
counts = pd.Series(y_seq).value_counts()
keep_classes = counts[counts >= 5].index.tolist()
mask = np.isin(y_seq, keep_classes)
X_seq = X_seq[mask]; y_seq = y_seq[mask]; last_idx = last_idx[mask]
print(f'After drop: {X_seq.shape[0]:,} sequences')

# Split (keep last_idx aligned)
X_train, X_tmp, y_train, y_tmp, idx_train, idx_tmp = train_test_split(
    X_seq, y_seq, last_idx, test_size=0.30, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test, idx_val, idx_test = train_test_split(
    X_tmp, y_tmp, idx_tmp, test_size=0.50, random_state=RANDOM_STATE)
print(f'Train {len(X_train):,}  Val {len(X_val):,}  Test {len(X_test):,}')


Sequences: (75651, 50, 77)
After drop: 75,644 sequences
Train 52,950  Val 11,347  Test 11,347


In [6]:
# Load 4 flow models
xgb_sess = ort.InferenceSession(XGB_ONNX, providers=['CPUExecutionProvider'])
lstm_sess = ort.InferenceSession(LSTM_ONNX, providers=['CPUExecutionProvider'])
trans_sess = ort.InferenceSession(TRANS_ONNX, providers=['CPUExecutionProvider'])
iso = joblib.load(IF_PKL)
print('4 flow models loaded')

def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def flow_model_features(X_split, bs=1024):
    """Run 4 flow models in batches (avoids OOM), return (N, 46)."""
    outs = []
    for i in range(0, len(X_split), bs):
        xb = X_split[i:i + bs]
        last_flow = xb[:, -1, :].astype(np.float32)
        full_seq = xb.astype(np.float32)
        xgb_out = xgb_sess.run(None, {'features': last_flow})
        xgb_proba = xgb_out[1] if xgb_out[1].shape[1] == N_CLASSES else xgb_out[0]
        lstm_logits = lstm_sess.run(None, {'input': full_seq})[0]
        lstm_proba = softmax(lstm_logits)
        trans_logits = trans_sess.run(None, {'input': full_seq})[0]
        trans_proba = softmax(trans_logits)
        if_scores = iso.decision_function(last_flow.astype(np.float64))
        if_attack = -if_scores.reshape(-1, 1)
        outs.append(np.concatenate([xgb_proba, lstm_proba, trans_proba, if_attack], axis=1).astype(np.float32))
    return np.concatenate(outs, axis=0)

# free RAM: these big arrays are not needed after the split
import gc
for _v in ['X', 'X_seq', 'X_tmp']:
    if _v in globals():
        del globals()[_v]
gc.collect()

print('Building flow-model features...')
F_train = flow_model_features(X_train)
F_val = flow_model_features(X_val)
F_test = flow_model_features(X_test)
print(f'Flow features: train {F_train.shape}, val {F_val.shape}, test {F_test.shape}')


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ExtraTreeRegressor from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator IsolationForest from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


4 flow models loaded
Building flow-model features...


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklea

Flow features: train (52950, 46), val (11347, 46), test (11347, 46)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(


In [7]:
# GNN: build graphs from generated CSV, run GAT, get per-IP node risk
class AttackPathGAT(torch.nn.Module):
    def __init__(self, in_channels=NODE_FEAT_DIM, hidden=GNN_HIDDEN, out_channels=1):
        super().__init__()
        self.g1 = GATConv(in_channels, hidden, heads=8, dropout=0.3)
        self.g2 = GATConv(hidden * 8, hidden, heads=4, dropout=0.3)
        self.g3 = GATConv(hidden * 4, out_channels, heads=1, concat=False)
    def forward(self, x, edge_index, edge_attr=None):
        x = F.elu(self.g1(x, edge_index))
        x = F.elu(self.g2(x, edge_index))
        x = self.g3(x, edge_index)
        return x.squeeze(-1)

gnn = AttackPathGAT().to(DEVICE)
gnn.load_state_dict(torch.load(GNN_PT, map_location=DEVICE))
gnn.eval()
print('GNN loaded')


GNN loaded


In [8]:
# Build IP -> node-risk map by running GNN on graphs
WINDOW_SECONDS = 5

def build_ip_risk_map(csv_path, max_rows=None):
    """Run GNN on all graphs, return dict IP -> mean node risk."""
    cols = [
        'Source IP', 'Destination IP', 'Source Port', 'Destination Port',
        'Protocol', 'Timestamp', 'Flow Duration',
        'Total Fwd Packets', 'Total Backward Packets',
        'Total Length of Fwd Packets', 'Total Length of Bwd Packets',
        'Average Packet Size', 'Max Packet Length',
        'SYN Flag Count', 'RST Flag Count', 'FIN Flag Count', 'ACK Flag Count',
        'Label'
    ]
    df = pd.read_csv(csv_path, usecols=cols, nrows=max_rows).dropna()
    df['is_attack'] = (df['Label'] != 'BENIGN').astype(int)
    df['t'] = pd.to_datetime(df['Timestamp'], format='%d/%m/%Y %H:%M:%S', errors='coerce')
    df['t'] = df['t'].fillna(pd.to_datetime(df['Timestamp'], format='%d/%m/%Y %H:%M', errors='coerce'))
    df = df.dropna(subset=['t'])
    df['t'] = df['t'].astype('datetime64[s]').astype('int64')
    df['window'] = df['t'] // WINDOW_SECONDS
    df['day'] = pd.to_datetime(df['t'], unit='s').dt.day
    df = df[df['day'] != 3]

    ip_risks = {}  # IP -> list of risk scores
    for _, group in df.groupby('window'):
        if len(group) < 2:
            continue
        ips = pd.unique(group[['Source IP', 'Destination IP']].values.ravel())
        ip_to_id = {ip: i for i, ip in enumerate(ips)}
        n = len(ips)
        feat = np.zeros((n, NODE_FEAT_DIM), dtype=np.float32)
        for ip, i in ip_to_id.items():
            m = (group['Source IP'] == ip) | (group['Destination IP'] == ip)
            sub = group[m]
            ports = pd.unique(sub[['Source Port', 'Destination Port']].values.ravel())
            peers = pd.unique(sub[['Source IP', 'Destination IP']].values.ravel())
            total_bytes = sub['Total Length of Fwd Packets'].sum() + sub['Total Length of Bwd Packets'].sum()
            total_pkts = sub['Total Fwd Packets'].sum() + sub['Total Backward Packets'].sum()
            dur = sub['Flow Duration'].sum()
            feat[i, 0] = len(sub)
            feat[i, 1] = total_bytes
            feat[i, 2] = total_pkts
            feat[i, 3] = sub['Average Packet Size'].mean()
            feat[i, 4] = len(ports)
            feat[i, 5] = len(peers)
            feat[i, 6] = sub['is_attack'].mean()
            feat[i, 7] = sub['Protocol'].nunique()
            feat[i, 8] = sub['SYN Flag Count'].sum()
            feat[i, 9] = sub['RST Flag Count'].sum()
            feat[i, 10] = sub['FIN Flag Count'].sum()
            feat[i, 11] = sub['ACK Flag Count'].sum()
            feat[i, 12] = dur
            feat[i, 13] = sub['Flow Duration'].mean()
            feat[i, 14] = sub['Max Packet Length'].max()
            feat[i, 15] = 1 if any(p in (80, 443) for p in ports) else (0.5 if 22 in ports else 0.0)

        edge_agg = group.groupby(['Source IP', 'Destination IP']).agg(
            pkt_count=('Total Fwd Packets', 'sum'),
            bytes_sum=('Total Length of Fwd Packets', 'sum'),
            proto=('Protocol', 'first'),
        ).reset_index()
        src_ids = edge_agg['Source IP'].map(ip_to_id).values
        dst_ids = edge_agg['Destination IP'].map(ip_to_id).values
        edge_index = torch.tensor([src_ids, dst_ids], dtype=torch.long).to(DEVICE)
        x = torch.tensor(feat, dtype=torch.float32).to(DEVICE)
        with torch.no_grad():
            scores = torch.sigmoid(gnn(x, edge_index)).cpu().numpy()
        for ip, i in ip_to_id.items():
            ip_risks.setdefault(ip, []).append(scores[i])

    return {ip: float(np.mean(r)) for ip, r in ip_risks.items()}

print('Building IP -> GNN risk map...')
ip_risk_map = build_ip_risk_map(GEN_CSV)
print(f'Mapped {len(ip_risk_map):,} IPs to GNN node risk')


Building IP -> GNN risk map...


/tmp/ipykernel_50/248681186.py:65: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  edge_index = torch.tensor([src_ids, dst_ids], dtype=torch.long).to(DEVICE)


Mapped 16,969 IPs to GNN node risk


In [9]:
# Map each sequence's last-flow Source IP -> GNN risk
gen_ip = pd.read_csv(GEN_CSV, usecols=['Source IP'])['Source IP'].values

def gnn_risk_for_idx(idx_arr):
    """For each original row index, return GNN risk of its Source IP."""
    risks = np.zeros(len(idx_arr), dtype=np.float32)
    for i, idx in enumerate(idx_arr):
        ip = gen_ip[idx]
        risks[i] = ip_risk_map.get(ip, 0.0)
    return risks.reshape(-1, 1)

g_train = gnn_risk_for_idx(idx_train)
g_val = gnn_risk_for_idx(idx_val)
g_test = gnn_risk_for_idx(idx_test)
print(f'GNN risks: train {g_train.shape}, val {g_val.shape}, test {g_test.shape}')
print(f'Non-zero GNN risk in test: {(g_test > 0).sum()} / {len(g_test)}')


GNN risks: train (52950, 1), val (11347, 1), test (11347, 1)
Non-zero GNN risk in test: 11266 / 11347


In [10]:
# Final fusion features = 46 flow + 1 GNN = 47
Z_train = np.concatenate([F_train, g_train], axis=1).astype(np.float32)
Z_val = np.concatenate([F_val, g_val], axis=1).astype(np.float32)
Z_test = np.concatenate([F_test, g_test], axis=1).astype(np.float32)
print(f'Fusion features: {Z_train.shape[1]} dims (46 flow + 1 GNN)')

class FusionMLP(nn.Module):
    def __init__(self, in_dim=FUSION_DIM, n_classes=N_CLASSES):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 64)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, n_classes)
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        return self.fc3(x)

fusion = FusionMLP().to(DEVICE)
optimizer = torch.optim.Adam(fusion.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

train_ds = TensorDataset(torch.tensor(Z_train), torch.tensor(y_train, dtype=torch.long))
val_ds = TensorDataset(torch.tensor(Z_val), torch.tensor(y_val, dtype=torch.long))
test_ds = TensorDataset(torch.tensor(Z_test), torch.tensor(y_test, dtype=torch.long))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
print(fusion)


Fusion features: 47 dims (46 flow + 1 GNN)
FusionMLP(
  (fc1): Linear(in_features=47, out_features=64, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=15, bias=True)
)


In [11]:
def evaluate_fusion(loader):
    fusion.eval()
    preds, labels = [], []
    with torch.no_grad():
        for zb, yb in loader:
            zb, yb = zb.to(DEVICE), yb.to(DEVICE)
            out = fusion(zb)
            preds.append(out.argmax(dim=1).cpu())
            labels.append(yb.cpu())
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()

for epoch in range(EPOCHS):
    fusion.train()
    total_loss = 0.0
    for zb, yb in train_loader:
        zb, yb = zb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        out = fusion(zb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    val_preds, val_labels = evaluate_fusion(val_loader)
    val_f1 = f1_score(val_labels, val_preds, average='macro', zero_division=0)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS}  loss={total_loss/len(train_loader):.4f}  val_f1={val_f1:.4f}')


Epoch   1/50  loss=0.8169  val_f1=0.6296
Epoch  10/50  loss=0.0013  val_f1=0.9844
Epoch  20/50  loss=0.0009  val_f1=0.9919
Epoch  30/50  loss=0.0008  val_f1=0.9857
Epoch  40/50  loss=0.0006  val_f1=0.9853
Epoch  50/50  loss=0.0005  val_f1=0.9857


In [12]:
test_preds, test_labels = evaluate_fusion(test_loader)
print('5-MODEL FUSION Results:')
print(f'  Accuracy : {accuracy_score(test_labels, test_preds):.4f}')
print(f'  F1 (macro): {f1_score(test_labels, test_preds, average="macro", zero_division=0):.4f}')
print(f'  F1 (weighted): {f1_score(test_labels, test_preds, average="weighted", zero_division=0):.4f}')
print()
print(classification_report(test_labels, test_preds, digits=3, zero_division=0))

print('=== Individual baselines (same test) ===')
xgb_out = xgb_sess.run(None, {'features': X_test[:, -1, :].astype(np.float32)})
xgb_pred = xgb_out[1].argmax(axis=1) if xgb_out[1].shape[1] == N_CLASSES else xgb_out[0].argmax(axis=1)
print(f'XGBoost macro F1: {f1_score(test_labels, xgb_pred, average="macro", zero_division=0):.4f}')
lstm_logits = np.concatenate([lstm_sess.run(None, {'input': X_test[i:i+1024].astype(np.float32)})[0] for i in range(0, len(X_test), 1024)], axis=0)
print(f'LSTM macro F1: {f1_score(test_labels, lstm_logits.argmax(axis=1), average="macro", zero_division=0):.4f}')
trans_logits = np.concatenate([trans_sess.run(None, {'input': X_test[i:i+1024].astype(np.float32)})[0] for i in range(0, len(X_test), 1024)], axis=0)
print(f'Transformer macro F1: {f1_score(test_labels, trans_logits.argmax(axis=1), average="macro", zero_division=0):.4f}')


5-MODEL FUSION Results:
  Accuracy : 0.9995
  F1 (macro): 0.9911
  F1 (weighted): 0.9995

              precision    recall  f1-score   support

           0      1.000     0.999     0.999      3053
           1      1.000     1.000     1.000        25
           2      1.000     1.000     1.000      1922
           3      1.000     1.000     1.000       157
           4      1.000     1.000     1.000      3430
           5      1.000     1.000     1.000        75
           6      1.000     1.000     1.000        91
           7      0.992     1.000     0.996       124
          10      0.999     1.000     0.999      2363
          11      1.000     0.987     0.993        77
          12      0.944     0.944     0.944        18
          14      0.923     1.000     0.960        12

    accuracy                          0.999     11347
   macro avg      0.988     0.994     0.991     11347
weighted avg      0.999     0.999     0.999     11347

=== Individual baselines (same test) ===
XG

In [13]:
torch.save(fusion.state_dict(), '/kaggle/working/fusion_mlp.pt')
print('Saved: /kaggle/working/fusion_mlp.pt')


Saved: /kaggle/working/fusion_mlp.pt
